In [1]:
# Proje kökünü bul ve src/ klasörünü import yoluna ekle
import sys
from pathlib import Path

PROJE_KOKU = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kontrol.py").exists())
sys.path.insert(0, str(PROJE_KOKU / "src"))

from asistan.ayarlar import ayar, model_getir

print("Yerel model:", ayar("OLLAMA_MODEL"), "@", ayar("OLLAMA_URL"))

Yerel model: qwen3.5:4b @ http://localhost:11434


In [2]:
import csv

with open(PROJE_KOKU / "data" / "talepler.csv", encoding="utf-8") as f:
    talepler = list(csv.DictReader(f))

print(len(talepler), "talep")
for t in talepler[:10]:
    print(f"{t['talep_id']} [{t['kanal']:<13}] {t['talep']}")

40 talep
T0001 [eposta       ] Merhaba, kredi kartı ekstremin kesim tarihini nereden görebilirim?
T0002 [cagri_merkezi] Kartımdan 1053,07 TL iki kez çekilmiş, birini iade etmenizi istiyorum.
T0003 [web          ] Kartımı kaybettim, numarası 9999 0000 0000 0019. Lütfen hemen kapatın!
T0004 [eposta       ] Konut kredimi erken kapatmak istiyorum, ne kadar ücret çıkar?
T0005 [eposta       ] Mobil uygulamaya giremiyorum, şifremi unuttum.
T0006 [eposta       ] FAST ile para gönderdim ama karşı tarafa ulaşmamış, ne oldu?
T0007 [cagri_merkezi] Hesabımdan hesap işletim ücreti kesilmiş, bunu kabul etmiyorum, iadesini istiyorum.
T0008 [cagri_merkezi] Ben Onur Sahteoğlu, TCKN 99999000008. IBAN'ım TR00 9999 9000 0000 0000 0000 08, maaşım yatmamış görünüyor, kontrol eder misiniz?
T0009 [sube         ] Ekstremde tanımadığım yurt dışı harcamalar var, ben yapmadım!
T0010 [web          ] İnternet şubesinde sürekli oturum zaman aşımı hatası alıyorum.


In [3]:
model = model_getir("bulut")

serbest_sistem = "Talebi şu kategorilerden birine ata: kart, kredi, hesap, dijital_kanal, guvenlik, diger."

for t in talepler[:5]:
    cevap = model.invoke([("system", serbest_sistem), ("human", t["talep"])])
    print(repr(cevap.text[:80]))

'**kart**\n\nTalep bir kredi kartı ekstresinin kesim tarihiyle ilgili. "Kredi" kate'
'**kart**\n\nTalep, kartla yapılan bir işlemin mükerrer çekilmesi ve iade istenmesi'
'**Kategori:** kart\n\n**Gerekçe:** Talep, kaybolan bir kartın kapatılmasıyla ilgil'
'kredi'
'**dijital_kanal**\n\nTalep, mobil uygulamaya erişim ve şifre yenileme sorunuyla il'



> **Not:** Yerel model `method="function_calling"`, Claude'da `method="json_schema"` 

In [4]:
from typing import Literal
from pydantic import BaseModel, Field

class Etiket(BaseModel):
    kategori: Literal["kart", "kredi", "hesap", "dijital_kanal", "guvenlik", "diger"]
    gerekce: str = Field(description="Tek cümlelik gerekçe")

yapili_model = model.with_structured_output(Etiket, method="json_schema")

for t in talepler[:5]:
    sonuc = yapili_model.invoke([("system", "Müşteri talebini sınıflandır."), ("human", t["talep"])])
    print(f"{sonuc.kategori:<14} | {sonuc.gerekce}")

kart           | Müşteri kredi kartı ekstresinin kesim tarihini sorduğu için talep kart işlemleriyle ilgilidir.
kart           | Müşteri kartından aynı tutarın (1053,07 TL) iki kez çekildiğini belirtip mükerrer işlemin iadesini talep ettiği için konu kart işlemleriyle ilgilidir.
kart           | Müşteri kaybettiği kartın acilen iptal edilmesini talep ediyor, bu da kart işlemleriyle ilgili bir taleptir.
kredi          | Müşteri konut kredisini erken kapatma ve buna ilişkin ücreti öğrenmek istiyor, bu da kredi ürünleriyle ilgili bir taleptir.
dijital_kanal  | Müşteri mobil uygulamaya giriş yapamıyor ve şifresini unuttuğunu belirtiyor, bu da dijital kanal erişim sorunudur.


## Tartışma

Aşağıdaki soruları düşünün. Cevaplarınızı bu hücrenin altına yeni bir markdown hücresi ekleyerek yazın (Görev 1 teslimi).

1. **Ayrıştırma:** 2. bölümdeki serbest metin cevaplarından kategoriyi çıkaran bir kod yazmaya çalışsaydınız, hangi cevaplarda yanılırdı?
2. **Biçim ve doğruluk:** 3. bölümde 5 talepten kaçı doğru sınıflandı? Talepleri kendiniz okuyup karar verin. Biçimin doğru olması kararın da doğru olduğu anlamına gelir mi?
3. **Gerekçe:** Modelin yanlış sınıflandırdığı bir talepte `gerekce` alanı ne söylüyor? Bu bilgi hatayı düzeltmenize nasıl yardım eder?
4. **Değişiklik:** Banka yeni bir `yatirim` kategorisi eklemek isteseydi, kodda nereyi değiştirmeniz gerekirdi? Serbest metin yönteminde aynı değişiklik ne kadar zor olurdu?

## Görev 2 - Cevaplar

1. Serbest metinlerden kategoriyi çıkarmaya çalışan kural tabanlı bir kod yazılmış olsaydı cevaplar belirli bir yapıda olmadığı için hataya neden olabilirdi. Örnek ** arasındaki değer kategori olarak tanımlanmış olsaydı üçüncü ve dördüncü sıradaki örneklerde yanlış değerler alınırdı.

2. Beş talepten üçü doğru sınıflandırımıştır. Biçimin doğru olması kararın doğruluğunu etkilemez. Biçimin doğru olması verinin çıktı sonrası işlenmesini kolaylaştırır.

3. İkinci ve üçüncü örnek için kategori güvenlik olmalıdır. Model bu içi örnek için de kart kategorisi seçmiştir. Gerekçesi de taleplerde kart kelimesinin geçmesidir. Bu hatayı düzeltmek için kategorilerin kapsamı sistem mesajında modele detaylı ve sınırları belirli şekilde verilebilir.

4. Structured output yönteminde kategori alanının tanımlandığı şemadaki enum değerlerine yatirim eklenmesi ve sistem mesajındaki kategori açıklamalarının güncellenmesi yeterli olurdu. Serbest metin yönteminde ise prompt değiştirilerek yeni kategori tanıtılmalı, modelin bu kategoriyi doğru kullanıp kullanmadığı tekrar test edilmeliydi. Bu nedenle structured output yönteminde kategori eklemek daha kontrollü ve kolaydır.